# Vrijeme do mergea PRa

## Uvod

### Pregled pojmova

**Git** je sustav za upravljanje verzijama programskog koda. Omogućava uvid u povijest svih promjena, vraćanje izmjena, kolaboriraju kroz grananje (branching) i spajanje (merging) te rješavaju konflikte u kodu.

**Pull Request (PR)** je mehanizam u Git verzioniranju koji omogućava programerima da predlože promjene u glavnu granu repozitorija. PR prolazi kroz proces pregleda i diskusije u svrhu poboljšanja, a zatim spajanja u glavnu granu.

### Izazovi

**Planiranje projekata**

Predviđanje vremena do spajanja (mergea) pomaže u boljem planiranju i raspodjeli resursa.

**Optimizacija procesa**

Prepoznavanje čimbenika koji utječu na brzinu mergea omogućuje optimizaciju razvojnog procesa.

**Povećanje produktivnosti**

Razumijevanje uzroka dugotrajnih PR-ova doprinosi kraćem ciklusu razvoja.

**Upravljanje očekivanjima**

Timovi mogu jasnije komunicirati očekivana vremena završetka PR-ova.

## Pregled projekta
Predvidjeti vrijeme do spajanja (mergea) GitHub Pull Requesta (PR-a) u efektivnim minutama, isključujući neradne dane i vikende, korištenjem metoda strojnog učenja.

### Okruženje

Podaci iz produkcijskih projekata, u kojima programeri rade u potpuno fleksibilnom radnom vremenu bez fiksnog radnog dana ili tjedna.

### Pristup
Prikupljanje podataka iz GitHub API-ja (vlastiti ), izrada i analizu značajki te iterativno poboljšavanje modela, s konačnim rješenjem temeljenim na ansambl pristupu s dva specijalizirana modela.

### Rezultati

Kroz 11 iterativnih faza razvoja, kreiran je ansambl model sa dva specializirana modela koji predviđaju vrijeme merge-a PR-ova u efektivnim minutama, uz implementaciju jednostavnog CLI koji automatski ekstrahira značajke iz GitHub API-ja i generira predikcije sa R² od 0.92.

Model pokazuje dobre performanse na training podacima, ali ograničena veličina dataset-a (233 PR-a) uzrokuje probleme s generalizacijom na novim podacima.


![Project Flowchart](diagrams/project_flowchart.png)

## Iteracije


### Faza 1: Prikupljanje i čišćenje podataka

**Cilj:** Prikupiti osnovne podatke o PR-ovima iz GitHub API-ja i pripremiti ih za daljnju analizu.

**Što se radilo:** Dohvaćanje PR podataka iz GitHub API-ja (pr.json → prs.json), čišćenje podataka (prs2.json) i strukturiranje u CSV format (prs_features.csv) kroz flattening kompleksne JSON strukture.

**Ishod:** Dobiven dataset sa 233 PR-a sa strukturiranim CSV-om koji sadrži osnovne feature-e: code metrics (additions, deletions, changed_files), review podaci (review_count, reviewers), timing informacije (created_at, merged_at) i tekstualne značajke (title, description). Target varijabla je postavljena na duration_hours (vrijeme od otvaranja do merge-a).



### Faza 2: Feature engineering

**Cilj:** Kreirati značajke koje bolje odražavaju realno vrijeme rada, uzimajući u obzir fleksibilno radno okruženje bez fiksnog radnog dana.

**Što se radilo:** Dodavanje efektivnog vremena koje isključuje vikende i praznike za Hrvatsku, analiza teksta (broj riječi u title/description), detekcija jezika (hrvatski vs engleski) u tekstu, te uklanjanje redundantnih stupaca i standardizacija naziva.

**Ishod:** Novi feature-i uključuju `effective_hours`, `effective_minutes`, `non_working_hours` za realističnije predikcije, te tekstualne značajke poput `title_word_count`, `description_word_count` i language percentages. Target varijabla je promijenjena na efektivne minute, što je ključno za realistične predikcije u fleksibilnom radnom okruženju.



### Faza 3: Analiza važnosti feature-a

**Cilj:** Identificirati koje značajke su najvažnije za predikciju vremena merge-a kako bi se fokusirao na relevantne prediktore.

**Što se radilo:** Analiza feature importance koristeći random forest algoritam, identifikacija najvažnijih prediktora i rangiranje feature-a po važnosti.

**Ishod:** 
- **Top 3 feature-a:** `non_working_hours` (RF importance: 0.33), `videos_count` (0.30), `title_croatian_pct` (0.16)
- Model je vrlo osjetljiv na vrijeme izvan rada i na medijski format (PR-ovi s videom) te na jezik naslova
- Ovo je postavilo osnovu za feature selection u kasnijim fazama i pomoglo razumjeti koje značajke model najviše koristi za predikciju

![Top 10 feature importance](faza_3/faza_3_feature_importance.png)



### Faza 4: Eksplorativna analiza podataka

**Cilj:** Razumjeti strukturu podataka, identificirati probleme poput redundantnih feature-a i outlier-a prije modeliranja.

**Što se radilo:** Korelacijska analiza između feature-a, analiza distribucija i outlier-a, analiza nedostajućih podataka, te kreiranje vizualizacija (boxplots, scatter plots, correlation matrices).

**Ishod:** 
- **Visoke korelacije:** `description_length` ↔ `description_word_count` = 0.99 (praktički isti signal)
- **Multikolinearnost u code metrics:** `additions` ↔ `total_lines_changed` = 0.99, `deletions` ↔ `total_lines_changed` = 0.97, `additions` ↔ `deletions` = 0.94
- Više parova feature-a ima korelaciju > 0.95, pa ih u kasnijim fazama reduciramo
- Detektirani outlier-i i asimetrične distribucije zahtijevaju dodatno čišćenje dataset-a prije modeliranja

![Top 10 korelacije](faza_4/faza_4_top_correlations.png)



### Faza 5: Ponovna analiza feature importance

**Cilj:** Potvrditi stabilnost feature importance nakon čišćenja dataset-a i osigurati da su najvažnije značajke identificirane.

**Što se radilo:** Ponovna analiza feature importance na pročišćenom dataset-u, usporedba važnosti feature-a prije i nakon čišćenja, te validacija stabilnosti feature importance.

**Ishod:** 
- **Top feature postaje:** `is_new_feature` (RF importance: 0.50) – novi/feature PR-ovi znatno mijenjaju trajanje
- **Novi top 5:** `images_count`, `title_croatian_words`, `description_croatian_words` ulaze u top 5 – vizualni i jezični dio PR-a se pokazuje bitan
- Nakon čišćenja podataka i novih feature-a, `is_new_feature` i `images_count` ulaze u top feature-e, što upućuje da su novi feature PR-ovi s puno slika najrizičniji za duže trajanje

![Feature importance nakon čišćenja](faza_5/faza_5_feature_importance_cleaned.png)



### Faza 6: Usporedba algoritama

**Cilj:** Pronaći najbolji algoritam za ovaj problem i implementirati osnovne tehnike feature engineeringa za poboljšanje performansi.

**Što se radilo:** Usporedba različitih algoritama (XGBoost, random forest, linear regression), feature engineering (ratio feature-i, polynomial feature-i, log transformacije), feature selection (SelectKBest), te hyperparameter tuning.

**Ishod:** 
- Test R² poboljšan sa 0.1148 na 0.1988 (poboljšanje od 73%), s test RMSE od 2048.26 minuta i test MAE od 1177.27 minuta
- **Top 3 feature-a za XGBoost:** `time_to_first_approval_minutes` (0.23), `commits` (0.14), `author_id` (0.09)
- Dodatno visoki: `comments`, `review_count`, `reviewer_count`, `changed_files`, `total_lines_changed`
- **Interpretacija:** koliko brzo dođe prvo odobrenje + koliko puta se PR mijenja (commits/lines/files) najviše objašnjava trajanje
- XGBoost je pokazao najbolje performanse, ali overfitting je bio vidljiv (velika razlika između train i test performansi)

![Usporedba algoritama](faza_6/faza_6_algorithm_comparison.png)


> **Hyperparameter tuning** je postupak u kojem tražimo najbolje "postavke" modela (npr. koliko je duboko stablo ili koliki je korak učenja) tako da model što bolje uči iz podataka, ali bez da ih samo "nabuba napamet".


### Faza 7: Poboljšanje kvalitete podataka

**Cilj:** Poboljšati kvalitetu i reprezentativnost dataset-a kako bi model bolje generalizirao na različite tipove PR-ova.

**Što se radilo:** Uklanjanje redundantnih stupaca (author_id, duplikati), balansiranje dataset-a po tehnologiji (frontend/backend/fullstack), balansiranje po tipu PR-a (fix/update/feature), te ekstrakcija reviewer feature-a iz reviews stupca.

**Ishod:** Pročišćen i balansiran dataset s dodanim reviewer feature-ima (`reviewer_count`, `reviewer_experience`) i poboljšanom reprezentativnošću različitih tipova PR-ova. Balansiran dataset poboljšava generalizaciju modela, a reviewer feature-i su se pokazali kao važni prediktori za predviđanje vremena merge-a.

![Distribucija podataka](faza_7/faza_7_data_distribution.png)



### Faza 8: XGBoost optimizacija

**Cilj:** Optimizirati XGBoost hiperparametre kako bi se postigle najbolje moguće performanse uz smanjenje overfitting-a.

**Što se radilo:** Optimizacija XGBoost hiperparametara (learning_rate, max_depth, n_estimators), analiza korelacija novih feature-a, detaljna analiza performansi modela, te primjena regularizacije za smanjenje overfitting-a.

**Ishod:** 
- Poboljšanje performansi - test R² sa 0.3749 na 0.9057 (poboljšanje od 142%), s test RMSE od 1177.58 minuta i test MAE od 774.22 minuta
- **Top feature-i nakon optimizacije:** `reviewer_team_size` (0.15), `reviewer_count` (0.08), `total_lines_changed_squared` (0.06), `time_to_first_approval_minutes_squared` (0.06)
- **Interpretacija:** nakon optimizacije model posebno gleda strukturu review tima i kvadratne (nelinearne) efekte `lines_changed` i vremena do prvog approvala
- Značajno smanjenje overfitting-a pokazalo je da je XGBoost prikladan izbor za ovaj problem i da optimizacija hiperparametara uz regularizaciju daje bolje rezultate

![Poboljšanje nakon optimizacije](faza_8/faza_8_optimization_improvement.png)


> **Post-processing calibration** je dodatno podešavanje izlaza modela (npr. vjerojatnosti ili vremena) nakon treniranja kako bi bolje odgovarali stvarnim ishodima.
>
> **Quantile regression** je vrsta regresije kojom ne predviđamo samo prosječnu vrijednost, nego određeni dio raspodjele (npr. donjih 25 % ili gornjih 10 % vrijednosti), što pomaže razumjeti ekstremnije slučajeve.
>
> **Sample weighting** je tehnika u kojoj nekim primjerima u skupu podataka dajemo veću, a nekima manju "važnost" pri učenju modela, kako bi se model više fokusirao na važnije ili rjeđe slučajeve.


### Faza 9: Napredna poboljšanja

**Cilj:** Poboljšati performanse modela, posebno za dugotrajne PR-ove, i dodati mogućnost predviđanja intervala pouzdanosti.

**Što se radilo:** Post-processing calibration za Long segment PR-ova, quantile regression za dinamičke prediction intervale, testiranje sample weighting strategija, te analiza grešaka po segmentima.

**Ishod:** 
- **Post-processing calibration:** R² 0.9057 → 0.9244, RMSE 1177.6 → 1054.7 min, MAE 774.2 → 729.9 min, Long MAE 1774.7 → 1603.7 min
- **Long segment bias:** model u prosjeku podcjenjuje duge PR-ove za ~1167 minuta
- **Intervals:** fiksna širina 4585.5 min → uvedeni dinamički intervali (5th–95th quantile) s prosječnom širinom ~3493.6 min
- Post-processing calibration je značajno poboljšao performanse, posebno za Long segment PR-ove koji zahtijevaju poseban tretman

![Napredna poboljšanja](faza_9/faza_9_advanced_improvements.png)



### Faza 10: Ensemble model

**Cilj:** Implementirati ensemble pristup sa specializiranim modelima za različite tipove PR-ova kako bi se postigle optimalne performanse za sve segmente.

**Motivacija za ensemble:**
- U Fazi 9 vidimo da model sustavno podcjenjuje duge PR-ove za ~1167 minuta (Long segment MAE 1603.7 min)
- Normalni model na velikim PR-ovima čak daje negativan R² i ima RMSE ≈ 4509 min; to znači da linearno poboljšavanje istog modela više ne pomaže
- Distribucija trajanja je multimodalna: kratki PR-ovi i dugi PR-ovi traže različite signale. Jedan model istovremeno loše pokriva obje skupine
- Zbog tog jasnog segment-biasa uvodimo drugi, specijalizirani model za duge PR-ove i spajamo ih u ensemble

**Što se radilo:** Implementacija ensemble pristupa sa dva specializirana modela - normal model za PR-ove ≤ 2880 minuta (75. percentil) i long model za PR-ove > 2880 minuta, automatski odabir modela na temelju threshold-a, te treniranje odvojenih modela za različite segmente.

**Ishod:** 
- **Normal model (≤ 2880 min):** R² ≈ 0.30, RMSE ≈ 734 min, MAE ≈ 555 min
- **Long model (> 2880 min):** R² ≈ 0.47, RMSE ≈ 4509 min, MAE ≈ 2853 min (uz 41%/42% bolje RMSE/MAE u odnosu na normalni model na tim istim velikim PR-ovima)
- **Ensemble (sve PR-ove):** R² ≈ 0.66, RMSE ≈ 2222 min, MAE ≈ 1066 min
- Ensemble automatski bira model po thresholdu; rezultat: na velikim PR-ovima RMSE/MAE poboljšani ~41/42%, a ukupno R² ensemblea ≈ 0.66 uz RMSE ≈ 2222 min i MAE ≈ 1066 min
- Time rješavamo glavni deficit iz Faze 9 (bias na long segment) bez žrtvovanja performansi na kratkim PR-ovima

![Usporedba ensemble modela](faza_10/faza_10_ensemble_comparison.png)



### Faza 11: Production-ready rješenje

**Cilj:** Kreirati gotov sustav koji omogućava jednostavnu upotrebu modela za predikciju vremena merge-a bilo kojeg PR-a iz GitHub API-ja.

**Što se radilo:** Kreiranje kompletnog predikcijskog sustava s implementacijom fetch PR podataka iz GitHub API-ja, feature extraction pipeline-a, model serialization i loading, te command-line interface za predikcije.

**Ishod:** 
- Production pipeline u Fazi 11 koristi ista optimizirana XGBoost i ensemble logiku (normal vs long model), skriveno iza CLI-ja `predict_pr.py`, tako da korisnik ne mora znati detalje modela
- Sustav je jednostavan za upotrebu - samo GitHub token i PR URL su potrebni

Primjer korištenja sustava za predikciju vremena merge-a PR-ova:

**Kroz CLI:**

```bash
python3 faza_11/predict_pr.py https://github.com/[USER]/[REPO]/pull/[PR_ID] github_pat_[TOKEN]
```

Sustav automatski:
- dohvaća podatke iz GitHub API-ja
- ekstrahira značajke
- bira odgovarajući model (normal ili long)
- generira predikciju u efektivnim minutama

